# DSAA — DBSCAN Tuning (v3)

**මොකද කරන්නේ:** දැනට තියෙන DBSCAN settings එකෙන් silhouette **0.2387** විතරයි (v2 එකේ 0.3587). Notebook එකම warning එකක් දාලා තියෙනවා:

> `⚠️ Weak separation — consider tuning eps or using cosine metric.`

මේ notebook එකෙන් `eps` සහ `metric` දෙකම sweep කරලා **හොඳම settings** හොයනවා.

**Cosine ඇයි:** fingerprint එකක row එකක් **1.0ට sum වෙනවා** (compositional / proportion vector). එහෙම data වලට Euclidean නෙවෙයි, **cosine** තමයි නිවැරදි metric එක. ඒක standard argument එකක් — panel එකට defend කරන්න ලේසියි.

**වැදගත්:** මේ notebook එකෙන් Drive එකට **මොකුත් ලියන්නේ නෑ**. Settings හොයනවා විතරයි. ඒවා main notebook එකට දාලා තමයි results generate කරන්නේ.

**වෙලාව:** ~විනාඩි 5

In [1]:
# ============================================================
# CELL 1: Drive mount + load saved fingerprints
# ============================================================
import numpy as np
from google.colab import drive
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

drive.mount('/content/drive')

dsaa_dir = '/content/drive/MyDrive/DeepSentinel/DeepSentinel_DSAA_v3'

d = np.load(f'{dsaa_dir}/fingerprints.npz', allow_pickle=True)
print('npz එකේ තියෙන arrays:', list(d.keys()))

fingerprints   = d['fingerprints']
cluster_labels = d['cluster_labels']   # දැනට තියෙන labels (සැසඳීමට)

n_cur   = len(set(cluster_labels[cluster_labels != -1]))
noise_c = int((cluster_labels == -1).sum())

print(f'\nFingerprints shape : {fingerprints.shape}   '
      f'(13 Signal-1 + 16 Signal-2 = 29 dims)')
print(f'Row sums (මුල් 5)  : {fingerprints[:5].sum(axis=1).round(4)}')
print(f'\nදැනට තියෙන්නේ    : euclidean, eps=0.1075 -> {n_cur} clusters, '
      f'noise={noise_c}, silhouette=0.2387')

Mounted at /content/drive
npz එකේ තියෙන arrays: ['fingerprints', 'signal_1', 'signal_2', 'cluster_labels', 'fraud_type']

Fingerprints shape : (8213, 29)   (13 Signal-1 + 16 Signal-2 = 29 dims)
Row sums (මුල් 5)  : [2. 2. 2. 2. 2.]

දැනට තියෙන්නේ    : euclidean, eps=0.1075 -> 12 clusters, noise=505, silhouette=0.2387


In [2]:
# ============================================================
# CELL 2: eps x metric sweep
# ============================================================
# silhouette එක විතරක් බලලා තෝරන්න එපා — cluster ටිකක් හදලා ඉතුරු
# ඔක්කොම noise කරොත් silhouette එක බොරුවට උස්සනවා. ඒ නිසා noise% සහ
# ලොකුම cluster එකේ % එකත් print කරනවා.
# ============================================================
MIN_SAMPLES = 10          # main notebook එකේ පාවිච්චි කරන අගයම
N = len(fingerprints)

rows = []
for metric, grid in [('euclidean', np.arange(0.05, 0.31, 0.01)),
                     ('cosine',    np.arange(0.02, 0.31, 0.01))]:
    for eps in grid:
        lab  = DBSCAN(eps=float(eps), min_samples=MIN_SAMPLES,
                      metric=metric, n_jobs=-1).fit_predict(fingerprints)
        keep = lab != -1
        k    = len(set(lab[keep]))

        # පිළිගන්න පුළුවන් විසඳුම් විතරක්
        if k < 4 or k > 30 or keep.sum() < 0.5 * N:
            continue

        sizes   = np.bincount(lab[keep])
        biggest = sizes.max() / N
        if biggest > 0.85:            # එකම ලොකු cluster එකක් = වැඩක් නෑ
            continue

        s = silhouette_score(fingerprints[keep], lab[keep], metric=metric,
                             sample_size=3000, random_state=42)
        rows.append((float(s), metric, float(eps), int(k),
                     int((~keep).sum()), float(biggest)))

rows.sort(reverse=True)

print(f"{'#':>3} {'silhouette':>11} {'metric':<10} {'eps':>6} "
      f"{'clusters':>9} {'noise':>7} {'noise%':>7} {'ලොකුම%':>8}")
print('-' * 70)
for i, (s, m, e, k, n, big) in enumerate(rows[:15]):
    print(f'{i:>3} {s:>11.4f} {m:<10} {e:>6.3f} {k:>9} '
          f'{n:>7} {n/N*100:>6.1f}% {big*100:>7.1f}%')

print(f'\nසැසඳීමට — v3 දැන් : 0.2387  euclidean  eps=0.108  12 clusters')
print(f'සැසඳීමට — v2      : 0.3587  euclidean  eps=0.118  19 clusters')
print(f'\nවලංගු විසඳුම් {len(rows)}ක් හම්බුණා.')

  #  silhouette metric        eps  clusters   noise  noise%   ලොකුම%
----------------------------------------------------------------------
  0      0.6394 cosine      0.050         4      68    0.8%    49.3%
  1      0.6378 cosine      0.070         4      57    0.7%    49.3%
  2      0.6363 cosine      0.200         4       5    0.1%    49.5%
  3      0.6363 cosine      0.190         4       5    0.1%    49.5%
  4      0.6363 cosine      0.180         4       5    0.1%    49.5%
  5      0.6359 cosine      0.220         4       4    0.0%    49.5%
  6      0.6359 cosine      0.210         4       4    0.0%    49.5%
  7      0.6347 cosine      0.060         4      60    0.7%    49.3%
  8      0.6166 cosine      0.040         5      90    1.1%    47.7%
  9      0.6041 cosine      0.160         5       8    0.1%    49.3%
 10      0.6041 cosine      0.150         5       8    0.1%    49.3%
 11      0.6041 cosine      0.140         5       8    0.1%    49.3%
 12      0.6028 cosine      0.17

In [3]:
# ============================================================
# CELL 3: තෝරගත්ත settings එකෙන් preview එකක්
# ============================================================
# උඩ table එකේ '#' column එකේ number එක මෙතන දාන්න.
# 0 = හොඳම silhouette. වෙන එකක් ඕන නම් ඒකේ number එක දාන්න.
# ============================================================
CHOICE = 0

BEST_SIL, BEST_METRIC, BEST_EPS, BEST_K, BEST_NOISE, _ = rows[CHOICE]

lab  = DBSCAN(eps=BEST_EPS, min_samples=MIN_SAMPLES,
              metric=BEST_METRIC, n_jobs=-1).fit_predict(fingerprints)
keep = lab != -1

print('=' * 60)
print(f'තෝරගත්තේ : metric={BEST_METRIC}  eps={BEST_EPS:.4f}  '
      f'min_samples={MIN_SAMPLES}')
print('=' * 60)
print(f'  Clusters   : {BEST_K}      (දැන් 12 | v2 19)')
print(f'  Noise      : {BEST_NOISE} ({BEST_NOISE/N*100:.1f}%)   (දැන් 505 / 6.1%)')
print(f'  Silhouette : {BEST_SIL:.4f}   (දැන් 0.2387 | v2 0.3587)')

verdict = ('✅ v2 එකටත් වඩා හොඳයි' if BEST_SIL > 0.3587 else
           '🟡 දැන් තියෙන එකට වඩා හොඳයි, v2ට වඩා අඩුයි' if BEST_SIL > 0.2387 else
           '⚠️ දියුණුවක් නෑ')
print(f'  තීන්දුව    : {verdict}')

print('\nCluster sizes:')
for cid in sorted(set(lab)):
    c = int((lab == cid).sum())
    name = 'noise' if cid == -1 else f'cluster {cid}'
    print(f'  {name:<12} {c:>6} ({c/N*100:>5.1f}%)')

print('\n' + '=' * 60)
print('ඊළඟට: main notebook එකේ CELL 7 එකේ මේ lines දෙක වෙනස් කරන්න')
print('=' * 60)
print(f"EPS = {BEST_EPS:.4f}")
print(f"dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, "
      f"metric='{BEST_METRIC}', n_jobs=-1)")

තෝරගත්තේ : metric=cosine  eps=0.0500  min_samples=10
  Clusters   : 4      (දැන් 12 | v2 19)
  Noise      : 68 (0.8%)   (දැන් 505 / 6.1%)
  Silhouette : 0.6394   (දැන් 0.2387 | v2 0.3587)
  තීන්දුව    : ✅ v2 එකටත් වඩා හොඳයි

Cluster sizes:
  noise            68 (  0.8%)
  cluster 0      4052 ( 49.3%)
  cluster 1        18 (  0.2%)
  cluster 2      3933 ( 47.9%)
  cluster 3       142 (  1.7%)

ඊළඟට: main notebook එකේ CELL 7 එකේ මේ lines දෙක වෙනස් කරන්න
EPS = 0.0500
dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, metric='cosine', n_jobs=-1)


## ඊළඟට කරන්න ඕන දේ

1. උඩ CELL 3 එකේ print වුණු lines දෙක අරගන්න.
2. **`DeepSentinel_DSAA_Framework_V3.ipynb`** එකේ **CELL 7** එකට ගිහින්:
   - `EPS = suggested_eps` → `EPS = <අලුත් අගය>`
   - `metric='euclidean'` → `metric='<අලුත් metric>'`
3. `dbscan_config.json` එකේ `'metric': 'euclidean'` කියලා **hardcode** කරලා තියෙනවා (CELL 13) — ඒකත් වෙනස් කරන්න, නැත්නම් save වෙන config එක වැරදියි.
4. Main notebook එකේ **CELL 1 සිට 14 දක්වා Run all** කරන්න (~විනාඩි 40). Typology table, radar chart, dashboard ඔක්කොම අලුත් clustering එකෙන් ආපහු හැදෙනවා.

⚠️ CELL 7 විතරක් run කරලා නවත්තන්න එපා — cells 8–14 පරණ labels එක්ක තියෙනවා, එතකොට artifacts අතර නොගැලපීමක් එනවා.